# 🎬 AI Motion Transfer on Google Colab (A100 80GB)

Dự án **AI Motion Transfer** cho phép sao chép chuyển động từ video nguồn sang ảnh tham chiếu, giữ nguyên đặc trưng khuôn mặt và trang phục.

- **Mô hình cốt lõi:** Wan2.1-14B-I2V (Diffusion Transformer)
- **Kỹ thuật học:** LoRA (rank 128) + 3D Conv Pose Encoder + Multi-Objective Loss
- **Tối ưu tốc độ:** **Latent Consistency Distillation (LCD)** rút ngắn từ 25 bước xuống **4-8 bước** (~15-20s / video trên A100)
- **Giao diện:** Gradio Web UI tích hợp sẵn link public truy cập từ xa (`share=True`)

--- 
## 1. Kiểm tra phần cứng GPU (A100 80GB)
Đảm bảo bạn đã chọn runtime: **Runtime -> Change runtime type -> A100 GPU**.

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Device: {gpu_name}")
    print(f"Total VRAM: {vram_gb:.2f} GB")
    assert "A100" in gpu_name or vram_gb >= 16, "Khuyến nghị dùng GPU A100 hoặc >=16GB VRAM cho Wan2.1-14B!"

--- 
## 2. (Khuyến nghị) Kết nối Google Drive để lưu Checkpoints lâu dài
Google Colab sẽ tự ngắt kết nối sau vài giờ nếu không tương tác. Mount Google Drive giúp bạn lưu lại dữ liệu và model an toàn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục lưu trữ checkpoints trên Drive
!mkdir -p /content/drive/MyDrive/AI_Motion_Transfer/checkpoints
!mkdir -p /content/drive/MyDrive/AI_Motion_Transfer/checkpoints_distill

--- 
## 3. Cài đặt mã nguồn và môi trường (Dependencies)

In [ ]:
# Nếu bạn đã upload code lên GitHub, clone về Colab:
# !git clone https://github.com/YOUR_USERNAME/AntigravityProject.git /content/AntigravityProject
# %cd /content/AntigravityProject

# Hoặc nếu chạy trực tiếp trong thư mục dự án:
import os
if not os.path.exists('requirements.txt'):
    print('Vui lòng cd vào thư mục dự án chứa requirements.txt')
else:
    !pip install -q -r requirements.txt
    !pip install -q -e .
    print('✅ Cài đặt thư viện hoàn tất!')

--- 
## 4. Chuẩn bị dữ liệu huấn luyện (Data Preparation)
Tải video nhảy mẫu (ví dụ AIST++, Pexels hoặc thư mục video của bạn) và tự động trích xuất pose bằng DWPose.

In [ ]:
# Tải và trích xuất pose cho 500 video nhảy mẫu
!python prepare_data.py \
    --source aist++ \
    --output_dir ./data \
    --max_videos 500 \
    --target_fps 15 \
    --target_resolution 832x480

# Kiểm tra tính toàn vẹn của dữ liệu
!python prepare_data.py --validate --output_dir ./data

--- 
## 5. Giai đoạn 1: Huấn luyện Teacher Model (LoRA + PoseEncoder)
Huấn luyện mô hình cơ sở ở chế độ chuẩn 25 bước trên A100 (BF16, Gradient Checkpointing, Multi-Objective Loss chống méo hình).

In [ ]:
# Chạy huấn luyện (Lưu checkpoints vào Google Drive nếu đã mount)
!accelerate launch train.py \
    --config config/default.yaml \
    --output_dir /content/drive/MyDrive/AI_Motion_Transfer/checkpoints \
    --seed 42

--- 
## 6. Giai đoạn 2: Consistency Distillation (Nén xuống 8 bước ⚡)
Sau khi Teacher đã học xong, chạy Consistency Distillation để huấn luyện Student LoRA rút ngắn thời gian sinh video xuống **~15-20 giây**.

In [ ]:
!accelerate launch train_distill.py \
    --config config/default.yaml \
    --teacher_checkpoint /content/drive/MyDrive/AI_Motion_Transfer/checkpoints/final \
    --output_dir /content/drive/MyDrive/AI_Motion_Transfer/checkpoints_distill \
    --num_steps 8

--- 
## 7. Khởi chạy Web UI (Gradio Public Link)
Khởi chạy giao diện web. Gradio sẽ tạo 1 link `https://xxxx.gradio.live` công khai để bạn tải ảnh tham chiếu và video chuyển động lên tạo video ngay trên trình duyệt điện thoại hoặc máy tính.

In [ ]:
# Khởi chạy app với public URL
!python app.py

--- 
## 8. Chạy Inference thử nghiệm từ Code Python (Không cần UI)
Bạn cũng có thể sinh video trực tiếp bằng đoạn code ngắn bên dưới:

In [ ]:
from src.pipeline import MotionTransferPipeline

pipeline = MotionTransferPipeline(
    config_path="config/default.yaml",
    lora_path="/content/drive/MyDrive/AI_Motion_Transfer/checkpoints_distill/final/distilled_lora",
    pose_encoder_path="/content/drive/MyDrive/AI_Motion_Transfer/checkpoints_distill/final/pose_encoder.pt",
    is_distilled=True,
    device="cuda",
)

# Kích hoạt tăng tốc TeaCache
pipeline.enable_teacache(threshold=0.05)
pipeline.enable_flash_attention()

# Sinh video 8 bước trong ~15 giây
output_video = pipeline.run(
    ref_image="./demo_ref.jpg",
    driving_video="./demo_driving.mp4",
    output_path="./outputs/demo_fast_result.mp4",
    num_steps=8,
    guidance_scale=2.0,
    seed=42,
)
print(f"Đã tạo video thành công tại: {output_video}")